In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import glob

import torch
import torch.nn
from pathlib import Path
import pytorch_lightning as pl
from pytorch_lightning.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
)


from src.dataset import *
from src.lightning import *
from src.models import *
from src.params import *
from src.utils import *
from src.deployment import *
import torch_pruning as tp
import torch.quantization



In [2]:
path = CHECKPOINTS_DIR / "final" / "epoch=26-val_loss=0.677.ckpt"
files_dir = PROCESSED_DIR / "cropped" / "files"
metadata = pd.read_csv(DATA_DIR / "train.csv")
dm = BrainDataModule(metadata=metadata, spec_dir= files_dir, batch_size= 32, num_workers= 8, verbose= False)


#need to fuse model
fuse_list = [
    ["model.backbone.0.0.block.0", "model.backbone.0.0.block.1", "model.backbone.0.0.block.2"],
    ["model.backbone.0.1.block.0", "model.backbone.0.1.block.1", "model.backbone.0.1.block.2"],
    ["model.backbone.1.0.block.0", "model.backbone.1.0.block.1", "model.backbone.1.0.block.2"],
    ["model.backbone.1.1.block.0", "model.backbone.1.1.block.1", "model.backbone.1.1.block.2"],
    ["model.backbone.2.0.block.0", "model.backbone.2.0.block.1", "model.backbone.2.0.block.2"],
    ["model.backbone.2.1.block.0", "model.backbone.2.1.block.1", "model.backbone.2.1.block.2"],
]

In [3]:
#wrapper custom for quantization

class QuantWrapperCustom(nn.Module):
    def __init__(self, model):
        super().__init__()
        
        self.quant = torch.ao.quantization.QuantStub()
        self.model = model
        self.dequant = torch.ao.quantization.DeQuantStub()
        
    def forward(self,x):
        x = self.quant(x)
        x = self.model.backbone(x)
        x = self.model.head(x)
        x = self.dequant(x)
        return nn.functional.log_softmax(x, dim= 1)
        
    

In [4]:
# instantiate unpruned best model to get initial architecture
model = instantiate_model(path)

#instantitate dataloaders 
dm.setup()

dummy_input = torch.randn(1,4,100,25)
imp = tp.importance.MagnitudeImportance(p= 2)


results_quantize = {
    "sizes_before": [], "latency_before": [], "score_before": [],
    "pruning_ratio": [], "sizes_after": [], 
    "latency_after": [], "score_after": []
}


[setup] Loading splits from cache: /Users/pablorougerie/code/Projets/brain_waves/data/cache/5_at_seed_273_min_vote_1.json
[setup] Cache loaded — 5 folds found


In [5]:
for pruning_ratio in [0,0.1, 0.25, 0.4]:
    #define pruner
    model = instantiate_model(path)
    print(f"testing with pruning ratio {pruning_ratio}")
    pruner = tp.pruner.MagnitudePruner(model= model, example_inputs= dummy_input,
                                   importance = imp,
                                   pruning_ratio = pruning_ratio,
                                   ignored_layers= [model.backbone[0][0].block[0], 
                                                        model.head])

    #prune in place
    pruner.step()
    
    #load fine tuned weights for that pruned architecture into lightning module
    ckpt_list = list((CHECKPOINTS_DIR / f"pruning_{pruning_ratio}").glob("*ckpt"))

    #ckpt = ckpt_list[0] #only one checkpoint
    lit_model = BrainLightning.load_from_checkpoint(ckpt_list[0], model = model)
    
    # model efficience before quantization
    size_before, latency_before, score_before = quantized_model_efficience_report(lit_model.model, datamodule= dm)
    results_quantize["sizes_before"].append(size_before)
    results_quantize["latency_before"].append(latency_before)
    results_quantize["score_before"].append(score_before)
    results_quantize["pruning_ratio"].append(pruning_ratio)
    
    
    #quantization
    model_to_quant = lit_model.model
    model_to_quant.cpu().float().eval()

    model_wrapped = QuantWrapperCustom(model = model_to_quant)
    torch.backends.quantized.engine = "qnnpack"
    
    model_fused = torch.ao.quantization.fuse_modules(model_wrapped, fuse_list, inplace= False)
    model_fused.qconfig = torch.ao.quantization.get_default_qconfig("qnnpack")
    model_prepared  = torch.ao.quantization.prepare(model_fused)
    
    
        #calibration
    model_prepared.eval()
    cal_dataloader = dm.val_dataloader()
    with torch.no_grad():
        for x, _ in cal_dataloader:
            model_prepared(x.cpu())
            
    #conversion
    model_quantized = torch.ao.quantization.convert(model_prepared)
    size_after, latency_after, score_after = quantized_model_efficience_report(model_quantized, datamodule= dm)
    results_quantize["sizes_after"].append(size_after)
    results_quantize["latency_after"].append(latency_after)
    results_quantize["score_after"].append(score_after)
    
    
    

testing with pruning ratio 0
calculate model inference score

-----------------------------------
  Model Efficiency Report Quantization
-----------------------------------
  Model size (MB):     4.217
-----------------------------------
  Median latency (ms): 4.05
  Max latency (ms):    5.43
-----------------------------------
  KL score: 0.6754
-----------------------------------



/var/folders/dk/v6_xfc950bb08vyf1lm9v0x00000gn/T/ipykernel_92744/562958115.py:37: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_prepared  = torch.ao.quantization.prepare(model_fused)
/var/folders/dk/v6_xfc950bb08vyf1lm9v0x00000gn/T/ipykernel_92744/562958115.py:48: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations o

calculate model inference score

-----------------------------------
  Model Efficiency Report Quantization
-----------------------------------
  Model size (MB):     1.064
-----------------------------------
  Median latency (ms): 2.57
  Max latency (ms):    9.24
-----------------------------------
  KL score: 0.6759
-----------------------------------

testing with pruning ratio 0.1
calculate model inference score

-----------------------------------
  Model Efficiency Report Quantization
-----------------------------------
  Model size (MB):     3.442
-----------------------------------
  Median latency (ms): 3.76
  Max latency (ms):    11.06
-----------------------------------
  KL score: 0.6780
-----------------------------------

calculate model inference score

-----------------------------------
  Model Efficiency Report Quantization
-----------------------------------
  Model size (MB):     0.870
-----------------------------------
  Median latency (ms): 2.28
  Max latency (ms

In [6]:
results_df = pd.DataFrame(results_quantize)
results_df

,sizes_before,latency_before,score_before,pruning_ratio,sizes_after,latency_after,score_after
0,4.217013,4.049937,0.675397,0.00,1.063870,2.574042,0.675909
1,3.442416,3.759292,0.678048,0.10,0.870206,2.284416,0.678543
2,2.510287,3.995605,0.681209,0.25,0.637113,1.917333,0.681632
3,1.688022,3.454813,0.685242,0.40,0.431546,1.814833,0.686190


In [7]:
results_df.to_csv(DATA_DIR / "quantize_results.csv", index=False)